# Q8：Kaggle 分段训练

一次运行一个 job 的一个训练段，保留完整总预算、样本顺序和共同初始化。当前清单有 **40 个阶段**（5 个外部、35 个官方），共 **113,236 次更新**；本 notebook 不会自动连续启动全部任务。

**输入与环境**
1. 上传 `runs/kaggle_preparation/ntllm-kaggle.zip` 为私有 Dataset，添加为输入。
2. 添加模型 [Qwen3-8B-Base v1](https://www.kaggle.com/models/qwen-lm/qwen-3/Transformers/8b-base/1)。使用输入挂载，不下载权重。
3. 使用 T4 和已验证的 CUDA 环境，开启 Internet 安装固定版本的 Python 依赖。配套 `kernel-metadata.json` 记录历史成功镜像，默认关闭 GPU 用于准备；正式运行时启用 T4。`centered_class` 已完成 T4 首段 120 步验证；长程效果以后续结果为准。
4. 续跑时添加上一段**成功保存**的 notebook 输出，把其中 `ntllm` 目录填到 `PREVIOUS_OUTPUTS`。官方适配需要该臂完整外部终点。

先用 `ACTION="prepare"` 在 CPU 核对输入；正式运行设为 `"run"` 并启用 GPU。默认每段 3 小时，每 100 步或 900 秒保存。建议使用 **Save Version → Save & Run All** 保存完整输出；不要等待硬超时才保存。[Kaggle 会话与输出说明](https://www.kaggle.com/docs/notebooks)

In [ ]:
import os
from pathlib import Path

JOB = os.environ.get("NTLLM_JOB", "centered_class")
ACTION = os.environ.get("NTLLM_ACTION", "prepare")  # prepare 或 run
SEGMENT_SECONDS = 10800
CHECKPOINT_EVERY = 100
CHECKPOINT_SECONDS = 900
STOP_AFTER_UPDATES = 0  # 0 清除额外步数上限；总训练预算保持不变

BUNDLE_ZIP = os.environ.get("NTLLM_BUNDLE", "")  # 留空则寻找唯一 ntllm-kaggle.zip
BASE_MODEL = ""  # 留空则只寻找路径含 8b-base 的挂载模型
PREVIOUS_OUTPUTS = []  # 例如 ["/kaggle/input/上一段输出/ntllm"]

WORKSPACE = Path(os.environ.get("NTLLM_WORKSPACE", "/kaggle/working/ntllm"))
LOCAL_CHECK = os.environ.get("NTLLM_LOCAL_CHECK") == "1"  # 本地准备验证专用
if ACTION not in {"prepare", "run"}:
    raise ValueError("ACTION 必须是 prepare 或 run")
if not LOCAL_CHECK and WORKSPACE != Path("/kaggle/working/ntllm"):
    raise ValueError("Kaggle 续跑须保持固定工作目录")

## 1. 解包并核对输入

上传包只含当前代码、配置、所需训练表、证据记录和 tokenizer。旧运行目录、原始来源下载和模型权重不在包中。首次解包后核对所有文件；恢复训练须使用同一版本上传包。

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile


def digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


if not (WORKSPACE / "bundle_manifest.json").is_file():
    if BUNDLE_ZIP:
        candidates = [Path(BUNDLE_ZIP)]
    else:
        candidates = list(Path("/kaggle/input").rglob("ntllm-kaggle.zip"))
        if not candidates:
            candidates = [p.parent for p in Path("/kaggle/input").rglob("bundle_manifest.json")
                          if not (p.parent / "runs/session_result.json").exists()]
    if len(candidates) != 1 or not candidates[0].exists():
        raise FileNotFoundError(f"请添加唯一上传包，或在 BUNDLE_ZIP 填写 ZIP / 已解包 ntllm 目录：{candidates}")
    if WORKSPACE.exists() and any(WORKSPACE.iterdir()):
        raise ValueError("工作目录非空但缺少包清单；请检查已有文件")
    WORKSPACE.parent.mkdir(parents=True, exist_ok=True)
    if candidates[0].is_dir():
        shutil.copytree(candidates[0], WORKSPACE, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(candidates[0]) as archive:
            for member in archive.infolist():
                parts = Path(member.filename).parts
                if not parts or parts[0] != "ntllm" or ".." in parts or Path(member.filename).is_absolute():
                    raise ValueError(f"包内路径不合法：{member.filename}")
            archive.extractall(WORKSPACE.parent)

manifest = json.loads((WORKSPACE / "bundle_manifest.json").read_text())
for relative, expected in manifest["files"].items():
    path = WORKSPACE / relative
    if not path.is_file() or path.stat().st_size != expected["bytes"] or digest(path) != expected["sha256"]:
        raise ValueError(f"上传包内容不一致：{relative}")
BUNDLE_ID = digest(WORKSPACE / "bundle_manifest.json")
catalog = json.loads((WORKSPACE / "kaggle_jobs.json").read_text())
job = catalog["jobs"][JOB]
os.environ.update(PYTHONPATH=str(WORKSPACE / "src"), HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
print(f"已核对 {len(manifest['files'])} 个输入文件；job={JOB}，预算={job['updates']:,} 次更新")

## 2. 恢复同一实验

`centered_class` 和 `source_baseline` 分别建立两组共同初始化。运行同组的其他外部臂前，必须添加首臂输出。官方 job 名为 `off_off_fold0`、`on_on_fold0`、`source_baseline_fold0` 等，fold 为 0–4。

只恢复当前 job、其外部父阶段及共同初始化，避免把 40 个阶段的全部文件堆入同一个输出。所需历史段、快照和结果仍随该 job 保留。请给同一 job 选择最近一次输出，不混用两个不同进度的版本。

In [ ]:
needed_jobs = [JOB]
if job.get("parent"):
    needed_jobs.append(job["parent"])
restore_marker = WORKSPACE / "runs" / "kaggle_session.json"


def merge_tree(source, destination):
    if not source.exists():
        return
    for path in sorted(source.rglob("*")):
        if path.is_symlink():
            raise ValueError(f"恢复输入含符号链接：{path}")
        if not path.is_file():
            continue
        target = destination / path.relative_to(source)
        if target.exists():
            if digest(target) != digest(path):
                raise ValueError(f"两个恢复输入内容冲突：{target}")
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)


if not restore_marker.exists():
    for location in PREVIOUS_OUTPUTS:
        source = Path(location)
        if not (source / "bundle_manifest.json").is_file() or digest(source / "bundle_manifest.json") != BUNDLE_ID:
            raise ValueError(f"恢复输出必须来自同一上传包：{source}")
        merge_tree(source / "runs/common_initialization", WORKSPACE / "runs/common_initialization")
        for identifier in needed_jobs:
            for kind in ("jobs", "prepared"):
                relative = Path("runs") / kind / identifier
                merge_tree(source / relative, WORKSPACE / relative)
    restore_marker.parent.mkdir(parents=True, exist_ok=True)
    restore_marker.write_text(json.dumps({"bundle_id": BUNDLE_ID, "job": JOB,
                                         "restored_from": PREVIOUS_OUTPUTS}, indent=2))
else:
    saved = json.loads(restore_marker.read_text())
    if saved["bundle_id"] != BUNDLE_ID or saved["job"] != JOB or saved["restored_from"] != PREVIOUS_OUTPUTS:
        raise ValueError("任务或恢复输入已改变；请先保存已有输出，再在新会话使用新参数")
print("需要保留的训练链：", needed_jobs)

## 3. 安装固定依赖

保留镜像中的 CUDA Torch，使用 `uv` 安装其余固定版本。不同续跑段应保持同一环境、GPU 类型和计算 dtype。准备模式不加载模型。

In [ ]:
import importlib.metadata as metadata
import platform

if not LOCAL_CHECK:
    required_torch = "2.11.0+cu130"
    if metadata.version("torch") != required_torch:
        raise RuntimeError(f"本配方已验证 Torch {required_torch}，当前是 {metadata.version('torch')}；请使用配套镜像")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    uv = [sys.executable, "-m", "uv", "pip"]
    # 历史成功 HF 环境也移除了这两个未使用、会触发可选导入冲突的包。
    subprocess.run(uv + ["uninstall", "--system", "--python", sys.executable,
                         "torchao", "bitsandbytes"], check=True)
    subprocess.run(uv + ["install", "--system", "--python", sys.executable,
                         "-r", str(WORKSPACE / "configs/backends/requirements-hf.txt")], check=True)

packages = ("torch", "transformers", "peft", "accelerate", "tokenizers", "numpy", "pandas", "pyarrow")
versions = {name: metadata.version(name) for name in packages}
versions["python"] = platform.python_version()
if not LOCAL_CHECK:
    for location in PREVIOUS_OUTPUTS:
        previous = Path(location) / "runs/session_result.json"
        if previous.is_file():
            prior = json.loads(previous.read_text())
            if prior.get("action") == "run" and prior.get("environment") != versions:
                raise ValueError("恢复段的 Python/依赖版本不同；请沿用首段镜像与依赖")
print(json.dumps(versions, indent=2))

## 4. 对齐基座与冻结 tokenizer

Kaggle 模型包与冻结计划的 tokenizer 配置文件可能不同。这里保留原权重、结构和词表，显式使用计划中的 tokenizer；加载 GPU 前先核对实际实现。临时基座位于 `/kaggle/temp`，不把权重复制进输出。每次续跑重新建立相同映射。

In [ ]:
base = None
if ACTION == "run":
    if LOCAL_CHECK:
        raise RuntimeError("本地准备验证不执行训练")
    if BASE_MODEL:
        mounted_base = Path(BASE_MODEL)
    else:
        bases = []
        for path in Path("/kaggle/input").rglob("config.json"):
            try:
                value = json.loads(path.read_text())
            except (ValueError, OSError):
                continue
            if "8b-base" in path.parts and value.get("model_type") == "qwen3" and value.get("hidden_size") == 4096 and value.get("num_hidden_layers") == 36:
                bases.append(path.parent)
        if len(bases) != 1:
            raise ValueError(f"请明确挂载唯一 Qwen3-8B-Base：{bases}")
        mounted_base = bases[0]

    frozen_tokenizer = WORKSPACE / catalog["tokenizer_dir"]
    # 核对模型结构及词表对应关系；基座权重和 config 始终取自原模型输入。
    for name in ("config.json", "tokenizer.json", "vocab.json", "merges.txt"):
        if digest(mounted_base / name) != digest(frozen_tokenizer / name):
            raise ValueError(f"基座与冻结 tokenizer 不兼容：{name}")
    index_name = "model.safetensors.index.json"
    index = json.loads((mounted_base / index_name).read_text())
    weight_names = sorted(set(index["weight_map"].values()))
    base = Path("/kaggle/temp/ntllm-base")
    base.mkdir(parents=True, exist_ok=True)
    for name in ["config.json", index_name, *weight_names]:
        relative = Path(name)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("权重索引路径必须位于基座目录内")
        source, target = mounted_base / relative, base / relative
        if not source.is_file():
            raise FileNotFoundError(source)
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() or target.is_symlink():
            if not target.is_symlink() or target.resolve() != source.resolve():
                raise ValueError(f"已有基座映射不一致：{target}")
        else:
            target.symlink_to(source.resolve())
    tokenizer_names = ("tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt",
                       "special_tokens_map.json", "added_tokens.json", "chat_template.jinja")
    for name in tokenizer_names:
        source = frozen_tokenizer / name
        if source.is_file():
            shutil.copy2(source, base / name)
    preflight = """
import json,sys
from pathlib import Path
from transformers import AutoTokenizer
from ntllm.q8_tokens import tokenizer_identity
base,frozen,out=map(Path,sys.argv[1:])
identities=[]
for path in (base,frozen):
    tokenizer=AutoTokenizer.from_pretrained(str(path),local_files_only=True)
    tokenizer.padding_side='right'
    if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
    identities.append(tokenizer_identity(path,tokenizer))
    assert max(tokenizer.get_vocab().values()) < json.loads((base/'config.json').read_text())['vocab_size']
for key in ('files_identity_sha256','runtime','backend_tokenizer_sha256'):
    if identities[0][key] != identities[1][key]: raise ValueError('Tokenizer preflight mismatch: '+key)
record={'base_view':str(base),'frozen_tokenizer':str(frozen),'tokenizer_identity':identities[0],
        'weights':'Read-only links to original mounted model; original model config retained.'}
out.write_text(json.dumps(record,indent=2))
print('冻结 tokenizer 与实际训练 tokenizer 一致；基座权重保持原挂载文件。')
"""
    subprocess.run([sys.executable, "-c", preflight, str(base), str(frozen_tokenizer),
                    str(WORKSPACE / "runs/base_overlay.json")], cwd=WORKSPACE, check=True)


## 5. 查看任务并准备计划

重建的云端计划逐项核对原冻结计划的科学设置、样本顺序、损失权重和 token 身份。只重新绑定机器路径，不改变训练预算。`fusion_robustness` 未混入这 40 项任务。

In [ ]:
from IPython.display import display
import pandas as pd

summary = [{"job": key, "阶段": value["phase"], "更新数": value["updates"],
            "外部父阶段": value.get("parent") or "—"} for key, value in catalog["jobs"].items()]
display(pd.DataFrame(summary).set_index("job"))
command = [sys.executable, "-m", "ntllm.q8_kaggle", "--workspace", str(WORKSPACE)]
subprocess.run(command + ["prepare", JOB], cwd=WORKSPACE, check=True)

## 6. 执行一个训练段

`run` 会读取实际恢复点自动续跑；已完整完成的 job 不会重新训练。官方适配要求父阶段达到完整总预算并完成评价。达到段时限后正常退出仍可能是 `interrupted`，这表示需要继续同一个 job。

In [ ]:
if ACTION == "run":
    subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(), '请启用 CUDA GPU'; print(torch.cuda.get_device_name())"], check=True)
    subprocess.run(command + ["run", JOB, "--base", str(base),
                              "--segment-seconds", str(SEGMENT_SECONDS),
                              "--checkpoint-every", str(CHECKPOINT_EVERY),
                              "--checkpoint-seconds", str(CHECKPOINT_SECONDS),
                              "--stop-after-updates", str(STOP_AFTER_UPDATES)], cwd=WORKSPACE, check=True)
else:
    print("准备完成；未加载模型、未执行 GPU 训练。正式运行将 ACTION 改为 run。")

## 7. 保存进度

每个 job 跨段保留最近两个恢复 checkpoint；旧段与稳定准备计划字节一致的大输入副本会去重，并记录引用位置。研究快照、训练行、结果、日志和模型终点保留。下一段添加本次保存的输出，再选择同一 job。

Kaggle 当前文档给出的输出上限为 20 GB。启动前须留出 6 GB 新段空间，下面显示实际输出大小。若空间不足，先下载保存历史结果，再安排后续阶段；不要只保留最后一个 checkpoint 文件夹。

In [ ]:
state_file = WORKSPACE / "runs/jobs" / JOB / "state.json"
state = json.loads(state_file.read_text())
size = sum(p.stat().st_size for p in WORKSPACE.rglob("*") if p.is_file())
status = {"job": JOB, "action": ACTION, "training_status": state.get("status", "prepared"),
          "checkpoint": state.get("checkpoint"), "workspace_GB": round(size / 1e9, 3),
          "bundle_id": BUNDLE_ID, "environment": versions}
(WORKSPACE / "runs/session_result.json").write_text(json.dumps(status, indent=2))
print(json.dumps(status, ensure_ascii=False, indent=2))
if size >= 18e9:
    print("输出已接近 20 GB；先保存并下载历史产物，再规划下一段。")

准备模式的输出只能证明输入与计划一致，不能证明 GPU 长程稳定或模型效果。正式训练后，以 `config.json` 的状态、`metrics.json` 的成功更新数和评价结果为准。每折保留 OOF 预测，完成同一实验的五折后再使用项目比较入口汇总。